# Global Organ Donation & Transplantation — Data Analytics Case Study
**Dataset:** `final organ dataset(3).csv`  
**Purpose:** Explore donor supply, transplant activity, waiting-list pressure, survival outcomes, geographic differences, and living-donor eligibility.

> **Important data-quality note:** This is an observational dataset. Some rows may contain missing values, internally inconsistent percentages/rates, or unusual country–continent combinations. Results are descriptive, not clinical advice or causal proof. Validate source definitions before making policy recommendations.

## Contents
1. Data loading, quality checks, preprocessing and EDA
2. What the data tells us
3. Fifteen professional analysis questions, each with a plot and interpretation
4. Hypothesis testing
5. Machine-learning model with evaluation and justification
6. Decision-oriented conclusion and dashboard export

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                             f1_score, roc_auc_score, mean_absolute_error,
                             mean_squared_error, r2_score)
from IPython.display import display
pd.set_option("display.max_columns", None)
DATA_PATH = "final organ dataset.csv"  # Keep the CSV in the same folder as this notebook.
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Shape: (2138, 18)


,Country,Continent,Population_M,Year,Organ,Deceased_Donors,Living_Donors,Total_Donors,Donor_Rate_PMP,Deceased_Rate_PMP,Transplants,Living_Donor_Pct,Deceased_Donor_Pct,Waiting_List,Avg_Wait_Years_US,Survival_Rate_1Y_Pct,Survival_Rate_5Y_Pct,Can_Living_Donate
0,Brazil,Americas,243.6,2026,Heart,436.0,0.0,436.0,1.79,1.79,482.0,0.0,100.0,2056.0,0.5,90.0,76.6,No
1,Pakistan,Americas,1.0,2016,Pancreas,2.0,86.0,88.0,88.00,2.00,0.0,0.7,100.0,1004.0,1.4,87.4,72.6,Rare (partial)
2,Iran,Europe,90.9,2026,Pancreas,39.0,NaN,162.0,1.78,0.43,149.0,4.0,99.4,180.0,1.4,89.2,70.4,Rare (partial)
3,Mexico,Americas,132.4,2016,Lung,76.0,35.0,111.0,0.84,0.57,87.0,3.3,99.4,200.0,0.3,83.5,59.7,Rare (lobar)
4,India,Asia,1350.0,2020,Pancreas,21.0,348.0,369.0,0.27,0.02,359.0,1.6,100.0,4493.0,1.4,86.3,73.3,Rare (partial)


## 1. Preprocessing and EDA
We preserve the raw dataframe, standardize text categories, inspect missingness and duplicates, convert numeric fields safely, and use median/mode imputation only inside model pipelines to avoid leakage. For descriptive charts, missing values are excluded from the relevant calculation.

In [2]:
raw = df.copy()
print("Rows, columns:", df.shape)
print("\nData types:")
display(df.dtypes.astype(str).rename("dtype").to_frame())
print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].rename("missing_count").to_frame())
print("Exact duplicate rows:", df.duplicated().sum())
print("\nNumeric summary:")
display(df.describe(include=[np.number]).T.round(2))
# Preprocess
df = df.drop_duplicates().copy()
text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"": np.nan, "nan": np.nan, "None": np.nan})
for c in df.columns:
    if c not in text_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
print("After duplicate removal:", df.shape)

Rows, columns: (2138, 18)

Data types:


,dtype
Country,object
Continent,object
Population_M,float64
Year,int64
Organ,object
Deceased_Donors,float64
Living_Donors,float64
Total_Donors,float64
Donor_Rate_PMP,float64
Deceased_Rate_PMP,float64



Missing values:


,missing_count
Deceased_Rate_PMP,74
Waiting_List,74
Population_M,74
Survival_Rate_5Y_Pct,74
Survival_Rate_1Y_Pct,74
Deceased_Donors,74
Living_Donors,74
Total_Donors,74
Donor_Rate_PMP,74
Avg_Wait_Years_US,74


Exact duplicate rows: 6

Numeric summary:


,count,mean,std,min,25%,50%,75%,max
Population_M,2064.0,82.40,215.55,1.0,7.80,35.70,70.10,1532.7
Year,2138.0,2023.32,16.36,1978.0,2019.00,2023.00,2025.00,2218.0
Deceased_Donors,2064.0,110.72,431.06,0.0,4.00,18.00,67.25,7196.0
Living_Donors,2064.0,130.15,385.22,0.0,2.00,14.00,81.00,6056.0
Total_Donors,2064.0,239.76,726.40,0.0,12.00,50.00,178.25,11098.0
Donor_Rate_PMP,2064.0,23.39,196.09,0.0,0.66,2.06,5.95,5405.0
Deceased_Rate_PMP,2064.0,7.98,60.77,0.0,0.16,0.90,3.02,1785.0
Transplants,2064.0,463.45,1783.56,0.0,14.00,64.00,295.00,40472.0
Living_Donor_Pct,2064.0,10.64,14.20,0.0,0.40,1.80,23.00,39.7
Deceased_Donor_Pct,2064.0,89.27,14.24,58.9,77.00,98.10,99.60,100.0


After duplicate removal: (2132, 18)


In [3]:
# EDA chart 1: missingness
miss = df.isna().sum().sort_values(ascending=False)
fig = px.bar(x=miss.index, y=miss.values, labels={"x":"Field","y":"Missing values"},
             title="EDA 1 — Missing Values by Field")
fig.update_layout(xaxis_tickangle=-45); fig.show()
# EDA chart 2: year coverage
fig = px.histogram(df, x="Year", nbins=max(5, df["Year"].nunique()),
                   title="EDA 2 — Records by Year", color="Continent")
fig.show()
# EDA chart 3: distributions of key variables
key_cols = [c for c in ["Total_Donors","Transplants","Waiting_List","Donor_Rate_PMP",
                         "Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"] if c in df]
long = df[key_cols].melt(var_name="Metric", value_name="Value").dropna()
fig = px.box(long, x="Metric", y="Value", points="outliers", title="EDA 3 — Distribution and Outliers in Key Metrics")
fig.update_layout(xaxis_tickangle=-25); fig.show()


## 2. What we learned from the data
Use the generated summaries and plots to describe:
- **Coverage:** number of records, countries, continents, organs and years represented.
- **Completeness:** fields with the most missing values and whether missingness may bias comparisons.
- **Spread and outliers:** donor counts, waiting-list counts, donor rates and survival measures may have very different scales.
- **Comparability:** raw counts are influenced by population size; per-million rates are more appropriate for cross-country comparisons when definitions are consistent.
- **Interpretation limits:** this dataset alone cannot establish that donor type causes better survival or shorter waits. Definitions, collection methods and source provenance should be verified.

## 3. Fifteen professional data-analysis questions
Each question includes a visualization and a short interpretation guide. Interpretations are data-led: read the plotted rankings, medians, trends and correlations rather than assuming a direction in advance.

### Q1. How does total donor volume vary across continents?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [4]:
tmp = df.groupby("Continent", dropna=False)["Total_Donors"].sum().reset_index().sort_values("Total_Donors", ascending=False)
fig = px.bar(tmp, x="Continent", y="Total_Donors", color="Continent", title="Q1 — Continental Donor Volume")
fig.show()
print(tmp.to_string(index=False))

Continent  Total_Donors
     Asia      215973.0
 Americas      143004.0
   Europe      114418.0
   Africa        9427.0
  Oceania        7197.0
   europe        1260.0


### Q2. Which countries contribute the highest total donor counts?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [5]:
tmp = df.groupby("Country", dropna=False)["Total_Donors"].sum().nlargest(15).sort_values().reset_index()
fig = px.bar(tmp, x="Total_Donors", y="Country", orientation="h", title="Q2 — Top 15 Countries by Total Donor Count")
fig.show(); display(tmp.sort_values("Total_Donors", ascending=False))

,Country,Total_Donors
14,United States,90106.0
13,China,60434.0
12,Turkey,46621.0
11,India,37668.0
10,Brazil,24545.0
9,Pakistan,23076.0
8,Iran,21318.0
7,France,20318.0
6,Spain,18860.0
5,United Kingdom,16081.0


### Q3. How does donor rate per million differ by continent?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [6]:
tmp = df.groupby("Continent", dropna=False)["Donor_Rate_PMP"].median().sort_values(ascending=False).reset_index()
fig = px.bar(tmp, x="Continent", y="Donor_Rate_PMP", color="Continent", title="Q3 — Median Donor Rate per Million by Continent")
fig.show(); display(tmp)

,Continent,Donor_Rate_PMP
0,Europe,2.815
1,europe,2.800
2,Americas,1.970
3,Oceania,1.970
4,Asia,1.695
5,Africa,0.320


### Q4. How do deceased and living donor contributions compare?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [7]:
tmp = df[["Living_Donors","Deceased_Donors"]].melt(var_name="Donor Type", value_name="Count").dropna()
fig = px.box(tmp, x="Donor Type", y="Count", points="outliers", title="Q4 — Living vs Deceased Donor Counts")
fig.show()
print(tmp.groupby("Donor Type")["Count"].describe().round(2))

                  count    mean     std  min  25%   50%    75%     max
Donor Type                                                            
Deceased_Donors  2058.0  110.38  430.83  0.0  4.0  18.0  67.75  7196.0
Living_Donors    2058.0  129.45  383.86  0.0  2.0  14.0  81.00  6056.0


### Q5. Which organs account for the largest transplant volumes?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [8]:
tmp = df.groupby("Organ", dropna=False)["Transplants"].sum().sort_values(ascending=False).reset_index()
fig = px.bar(tmp, x="Organ", y="Transplants", color="Organ", title="Q5 — Total Transplants by Organ")
fig.update_layout(xaxis_tickangle=-30); fig.show(); display(tmp)

,Organ,Transplants
0,Kidney,582338.0
1,Liver,244898.0
2,Heart,61800.0
3,Lung,38481.0
4,Pancreas,23040.0
5,Intestine,5005.0


### Q6. Where is waiting-list pressure highest by organ?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [9]:
tmp = df.groupby("Organ", dropna=False)["Waiting_List"].median().sort_values(ascending=False).reset_index()
fig = px.bar(tmp, x="Organ", y="Waiting_List", color="Organ", title="Q6 — Median Waiting-List Size by Organ")
fig.update_layout(xaxis_tickangle=-30); fig.show(); display(tmp)

,Organ,Waiting_List
0,Kidney,3633.0
1,Liver,345.0
2,Heart,140.0
3,Lung,37.0
4,Pancreas,32.0
5,Intestine,8.0


### Q7. Is donor volume associated with transplant volume?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [10]:
tmp = df[["Total_Donors","Transplants","Organ"]].dropna()
fig = px.scatter(tmp, x="Total_Donors", y="Transplants", color="Organ", hover_data=["Organ"],
                 trendline="ols", title="Q7 — Total Donors vs Transplants")
fig.show()
print("Pearson correlation:", round(tmp["Total_Donors"].corr(tmp["Transplants"]), 3))

Pearson correlation: 0.702


### Q8. Is donor rate associated with waiting-list size?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [11]:
tmp = df[["Donor_Rate_PMP","Waiting_List","Continent"]].dropna()
fig = px.scatter(tmp, x="Donor_Rate_PMP", y="Waiting_List", color="Continent",
                 title="Q8 — Donor Rate per Million vs Waiting List")
fig.show()
print("Spearman correlation:", round(tmp["Donor_Rate_PMP"].corr(tmp["Waiting_List"], method="spearman"), 3))

Spearman correlation: 0.327


### Q9. How do one-year survival outcomes vary by organ?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [12]:
tmp = df.groupby("Organ", dropna=False)["Survival_Rate_1Y_Pct"].median().sort_values().reset_index()
fig = px.bar(tmp, x="Survival_Rate_1Y_Pct", y="Organ", orientation="h", color="Organ",
             title="Q9 — Median One-Year Survival by Organ")
fig.show(); display(tmp)

,Organ,Survival_Rate_1Y_Pct
0,Intestine,75.0
1,Lung,85.0
2,Heart,88.0
3,Pancreas,88.0
4,Liver,91.9
5,Kidney,97.0


### Q10. How do five-year survival outcomes vary by organ?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [13]:
tmp = df.groupby("Organ", dropna=False)["Survival_Rate_5Y_Pct"].median().sort_values().reset_index()
fig = px.bar(tmp, x="Survival_Rate_5Y_Pct", y="Organ", orientation="h", color="Organ",
             title="Q10 — Median Five-Year Survival by Organ")
fig.show(); display(tmp)

,Organ,Survival_Rate_5Y_Pct
0,Intestine,50.10
1,Lung,59.10
2,Pancreas,73.05
3,Heart,77.00
4,Liver,77.90
5,Kidney,90.10


### Q11. What is the observed difference between one-year and five-year survival?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [14]:
tmp = df[["Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"]].melt(var_name="Follow-up", value_name="Survival (%)").dropna()
fig = px.violin(tmp, x="Follow-up", y="Survival (%)", box=True, points="outliers",
                title="Q11 — Distribution of One-Year vs Five-Year Survival")
fig.show()
paired = df[["Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"]].dropna()
print("Mean 1Y:", round(paired.iloc[:,0].mean(),2), "| Mean 5Y:", round(paired.iloc[:,1].mean(),2))

Mean 1Y: 87.46 | Mean 5Y: 71.11


### Q12. How does the living-donor share vary by organ?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [15]:
tmp = df.groupby("Organ", dropna=False)["Living_Donor_Pct"].median().sort_values().reset_index()
fig = px.bar(tmp, x="Living_Donor_Pct", y="Organ", orientation="h", color="Organ",
             title="Q12 — Median Living-Donor Share by Organ")
fig.show(); display(tmp)

,Organ,Living_Donor_Pct
0,Heart,0.1
1,Pancreas,0.5
2,Intestine,1.0
3,Lung,1.9
4,Liver,23.0
5,Kidney,37.0


### Q13. How has transplant activity changed over time?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [16]:
tmp = df.groupby("Year", dropna=False)["Transplants"].sum().reset_index().sort_values("Year")
fig = px.line(tmp, x="Year", y="Transplants", markers=True, title="Q13 — Annual Transplant Volume")
fig.show(); display(tmp)

,Year,Transplants
0,1978,26246.0
1,2015,61967.0
2,2016,98255.0
3,2017,51815.0
4,2018,50644.0
5,2019,44193.0
6,2020,34451.0
7,2021,30400.0
8,2022,110845.0
9,2023,72363.0


### Q14. Which country-organ records show the greatest waiting-list burden relative to transplants?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [17]:
tmp = df[["Country","Organ","Waiting_List","Transplants"]].dropna().copy()
tmp["Waiting_List_to_Transplants"] = tmp["Waiting_List"] / tmp["Transplants"].clip(lower=1)
tmp = tmp.nlargest(20, "Waiting_List_to_Transplants").sort_values("Waiting_List_to_Transplants")
fig = px.bar(tmp, x="Waiting_List_to_Transplants", y="Country", color="Organ", orientation="h",
             hover_data=["Waiting_List","Transplants"], title="Q14 — Highest Waiting-List-to-Transplant Ratios (Top 20)")
fig.show(); display(tmp.tail(10))

,Country,Organ,Waiting_List,Transplants,Waiting_List_to_Transplants
1835,Iran,Kidney,22186.0,0.0,22186.0
1107,China,Liver,30331.0,0.0,30331.0
1271,Egypt,Kidney,43918.0,0.0,43918.0
1389,Brazil,Kidney,46349.0,0.0,46349.0
1347,Brazil,Kidney,46349.0,0.0,46349.0
1691,Pakistan,Kidney,106613.0,0.0,106613.0
1474,Pakistan,Kidney,106613.0,0.0,106613.0
1510,Pakistan,Kidney,118128.0,0.0,118128.0
576,China,Kidney,223396.0,0.0,223396.0
224,China,Kidney,362775.0,0.0,362775.0


### Q15. Does living-donor eligibility category align with observed living-donor share?
**Analysis and justification:** This view is selected to compare the relevant groups or examine the relationship between the variables. Use rates for population-adjusted comparisons and treat correlations as associations, not causation. After running the cell, record the highest/lowest groups, the pattern, any outliers, and one limitation in your report.

In [18]:
tmp = df[["Can_Living_Donate","Living_Donor_Pct"]].dropna()
fig = px.box(tmp, x="Can_Living_Donate", y="Living_Donor_Pct", color="Can_Living_Donate",
             title="Q15 — Living-Donor Share by Eligibility Category")
fig.update_layout(xaxis_tickangle=-20); fig.show()
display(tmp.groupby("Can_Living_Donate")["Living_Donor_Pct"].agg(["count","median","mean"]).round(2))

,count,median,mean
Can_Living_Donate,,,
No,348,0.1,0.48
Rare,349,1.0,1.11
Rare (lobar),339,1.9,1.99
Rare (partial),342,0.5,0.84
Yes,340,37.0,36.95
Yes (partial),340,23.0,22.96


## 4. Hypothesis testing
Use a 5% significance level (α = 0.05). The tests below provide statistical evidence for specific differences/associations, not proof of causality. Because multiple comparisons can inflate false positives, interpret p-values alongside effect sizes and data quality.

In [19]:
# Test A: Is the paired mean one-year survival different from five-year survival?
paired = df[["Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"]].dropna()
if len(paired) >= 2:
    tstat, pval = stats.ttest_rel(paired["Survival_Rate_1Y_Pct"], paired["Survival_Rate_5Y_Pct"])
    diff = (paired["Survival_Rate_1Y_Pct"] - paired["Survival_Rate_5Y_Pct"]).mean()
    print(f"H0: mean paired difference = 0 | H1: mean paired difference != 0")
    print(f"n={len(paired)}, mean(1Y-5Y)={diff:.3f} percentage points, t={tstat:.3f}, p={pval:.4g}")
    print("Decision:", "Reject H0" if pval < 0.05 else "Fail to reject H0")
# Test B: Does median donor rate differ across continents? Kruskal-Wallis robust to non-normality.
groups = [g["Donor_Rate_PMP"].dropna().values for _,g in df.groupby("Continent") if g["Donor_Rate_PMP"].notna().sum() >= 2]
if len(groups) >= 2:
    stat, pval = stats.kruskal(*groups)
    print(f"\nH0: donor-rate distributions are equal across continents; H1: at least one differs")
    print(f"Kruskal-Wallis H={stat:.3f}, p={pval:.4g}; decision:", "Reject H0" if pval < .05 else "Fail to reject H0")
# Test C: Is donor count associated with transplant count? Pearson correlation test.
pair = df[["Total_Donors","Transplants"]].dropna()
if len(pair) >= 3:
    r, pval = stats.pearsonr(pair["Total_Donors"], pair["Transplants"])
    print(f"\nH0: population correlation = 0; H1: correlation != 0")
    print(f"n={len(pair)}, r={r:.3f}, p={pval:.4g}; decision:", "Reject H0" if pval < .05 else "Fail to reject H0")
# Test D: Is waiting-list size different between records above and below median donor rate?
test_df = df[["Donor_Rate_PMP","Waiting_List"]].dropna().copy()
if len(test_df) >= 4:
    med = test_df["Donor_Rate_PMP"].median()
    low = test_df.loc[test_df["Donor_Rate_PMP"] <= med, "Waiting_List"]
    high = test_df.loc[test_df["Donor_Rate_PMP"] > med, "Waiting_List"]
    if len(low) and len(high):
        stat, pval = stats.mannwhitneyu(low, high, alternative="two-sided")
        print(f"\nH0: waiting-list distributions are equal across lower/higher donor-rate groups")
        print(f"Median wait low-rate={low.median():.2f}, high-rate={high.median():.2f}; U={stat:.1f}, p={pval:.4g}")
        print("Decision:", "Reject H0" if pval < .05 else "Fail to reject H0")


H0: mean paired difference = 0 | H1: mean paired difference != 0
n=1988, mean(1Y-5Y)=16.351 percentage points, t=98.647, p=0
Decision: Reject H0

H0: donor-rate distributions are equal across continents; H1: at least one differs
Kruskal-Wallis H=158.435, p=2.134e-32; decision: Reject H0

H0: population correlation = 0; H1: correlation != 0
n=1987, r=0.702, p=7.19e-295; decision: Reject H0

H0: waiting-list distributions are equal across lower/higher donor-rate groups
Median wait low-rate=34.00, high-rate=244.00; U=312088.5, p=2.346e-45
Decision: Reject H0


## 5. Machine-learning model and justification
**Task:** predict the `Can_Living_Donate` category from the remaining available fields.  
**Baseline model:** Random Forest classifier with preprocessing inside a pipeline. Random Forest handles nonlinear relationships and mixed numeric/categorical predictors, and offers feature-importance diagnostics. A random train/test split evaluates generalization to held-out rows, but if repeated country-organ-year records exist, a grouped/time-based split may be more realistic. This model is an educational pattern-finding exercise, **not** a clinical eligibility tool. Remove any target-leaking columns and verify labels before interpreting results.

In [20]:
target = "Can_Living_Donate"
ml = df.dropna(subset=[target]).copy()
# Exclude identifiers and avoid direct target proxies where appropriate; keep other observed columns.
X = ml.drop(columns=[target])
y = ml[target].astype(str)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Ensure categorical columns use standard object dtype and np.nan for scikit-learn SimpleImputer
for c in cat_cols:
    X[c] = X[c].astype(object).replace({pd.NA: np.nan})

pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
model = Pipeline([("preprocess", pre),
                  ("model", RandomForestClassifier(n_estimators=250, min_samples_leaf=2,
                                                   class_weight="balanced", random_state=42,
                                                   n_jobs=-1))])
if y.nunique() >= 2 and y.value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Macro F1:", round(f1_score(y_test, pred, average="macro"), 4))
    print("\nClassification report:\n", classification_report(y_test, pred, zero_division=0))
    labels = sorted(y.unique())
    cm = confusion_matrix(y_test, pred, labels=labels)
    fig = px.imshow(cm, x=labels, y=labels, text_auto=True, aspect="auto",
                    labels=dict(x="Predicted", y="Actual", color="Count"),
                    title="Random Forest — Confusion Matrix")
    fig.show()
    try:
        names = model.named_steps["preprocess"].get_feature_names_out()
        imp = model.named_steps["model"].feature_importances_
        importance = pd.DataFrame({"Feature": names, "Importance": imp}).nlargest(20, "Importance").sort_values("Importance")
        fig = px.bar(importance, x="Importance", y="Feature", orientation="h",
                     title="Random Forest — Top 20 Feature Importances")
        fig.show()
    except Exception as e:
        print("Feature importance chart unavailable:", e)
else:
    print("Not enough examples per target class for a stratified split. Review class counts before modeling.")
display(y.value_counts().rename("count").to_frame())

Accuracy: 1.0
Macro F1: 1.0

Classification report:
                 precision    recall  f1-score   support

            No       1.00      1.00      1.00        72
          Rare       1.00      1.00      1.00        71
  Rare (lobar)       1.00      1.00      1.00        71
Rare (partial)       1.00      1.00      1.00        71
           Yes       1.00      1.00      1.00        71
 Yes (partial)       1.00      1.00      1.00        71

      accuracy                           1.00       427
     macro avg       1.00      1.00      1.00       427
  weighted avg       1.00      1.00      1.00       427



,count
Can_Living_Donate,
No,358
Yes,357
Rare,356
Rare (lobar),355
Rare (partial),353
Yes (partial),353


## 6. Decision-oriented findings and conclusion
Complete this section after reviewing the executed charts and statistical tests:
1. Prioritize countries/regions with high waiting-list burden and low population-adjusted donor rates for further investigation.
2. Compare donor counts with transplant volumes, but investigate mismatches rather than assuming extra donors automatically translate into transplants.
3. Review organ-specific waiting-list and survival patterns separately; pooling organs can conceal important differences.
4. Treat annual trends cautiously if year coverage is uneven or the dataset includes projected/synthetic records.
5. Use hypothesis tests to support or qualify observed differences, reporting effect sizes, sample sizes and p-values.
6. Do not use the machine-learning output for real clinical decisions. Validate provenance, label definitions, missingness, fairness and temporal generalization first.

**Final conclusion template:** “The analysis indicates [key evidence]. The most decision-relevant pattern is [pattern], supported by [visualization/test and statistic]. Therefore, we recommend [action] for further evaluation, subject to [data limitation].”

In [21]:
# Export a cleaned, Tableau-friendly flat file for dashboard building.
export_path = "organ_donation_cleaned_tableau.csv"
df.to_csv(export_path, index=False)
print("Saved:", export_path, "| rows:", len(df), "| columns:", len(df.columns))


Saved: organ_donation_cleaned_tableau.csv | rows: 2132 | columns: 18
